In [43]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for candidate in (Path.cwd(), Path.cwd().parent):
    pymrm_src = candidate / "pymrm" / "src"
    if pymrm_src.exists() and str(pymrm_src) not in sys.path:
        sys.path.insert(0, str(pymrm_src))

from pymrm import (
    NumJac,
    construct_coefficient_matrix,
    construct_convflux_upwind,
    construct_div,
    construct_grad,
    newton,
    non_uniform_grid,
)

from config import ModelConfig
from buildoperators import TransportOperators


cfg = ModelConfig()
model = TransportOperators(cfg)

print(cfg.rho_bulk)

AttributeError: 'ModelConfig' object has no attribute 'rho_bulk'

In [36]:
STOICH = np.array(
    [
        [-1.0, -3.0,  1.0,  1.0,  0.0],  # R1: CO2 + 3H2 <-> CH3OH + H2O
        [-1.0,  0.0, -2.0,  1.0,  1.0],  # R2: CO2 + 2CH3OH <-> DMC + H2O
    ]
)

SPECIES_LABELS = ("CO2", "H2", "CH3OH", "H2O", "DMC")

def reaction_rates(c_p: np.ndarray, cfg: ModelConfig) -> tuple[np.ndarray, np.ndarray]:
    RT = cfg.R * cfg.T
    P_Pa = c_p * RT                  # partial pressures [Pa], shape (n_z, n_r_ret, n_c)

    P_CO2_bar = P_Pa[..., 0] / 1e5
    P_H2_bar = P_Pa[..., 1] / 1e5
    P_CH3OH_bar = P_Pa[..., 2] / 1e5
    P_H2O_bar = P_Pa[..., 3] / 1e5
    P_DMC_bar = P_Pa[..., 4] / 1e5      
    P_total_Pa = P_Pa.sum(axis=-1)

    # R1: CO2 + 3H2 <-> CH3OH + H2O
    alpha_1 = P_CO2_bar * P_H2_bar**3 - (P_CH3OH_bar * P_H2O_bar) / cfg.k1_eq
    inhibition = (1.0 + cfg.K_ads(cfg.K_CO2_ref, cfg.dH_CO2) * P_CO2_bar + np.sqrt(cfg.K_ads(cfg.K_H2_ref, cfg.dH_H2) * P_H2_bar)) ** 2
    r1 = np.zeros_like(alpha_1)
    valid_idx = P_H2_bar > 1e-8
    r1 = (cfg.k_eff_r1() * alpha_1 / (P_H2_bar ** 2 * inhibition)) 
    r1 = r1 * cfg.rho_bulk()  # [mol/s/bar²/kg_cat] → [mol/s/m³_reactor]   

    # R2: CO2 + 2CH3OH <-> DMC + H2O  (Ibrahim et al., Eq. 6, no adsorption terms)
    alpha_2 = np.where(P_CO2_bar > 1e-8, P_DMC_bar / P_CO2_bar, 0.0)
    r2 = cfg.k_eff_r2(P_total_Pa) * ((1.0 - alpha_2) ** 3 - P_DMC_bar**2 / cfg.k2_eq)
    r2 = r2 * cfg.rho_bulk()  # zelfde omrekening
    
    return r1, r2

def particle_reaction_rates(c_p: np.ndarray, cfg: ModelConfig) -> np.ndarray:
    r1, r2 = reaction_rates(c_p, cfg)
    rates = np.stack([r1, r2], axis=-1)             
    return np.einsum("zrc,co->zro", rates, STOICH)  

class MembraneReactorModel:
    species_labels = SPECIES_LABELS
    stoich = STOICH

    def __init__(self, cfg: ModelConfig) -> None:
        self.cfg = cfg
        self.ops = TransportOperators(cfg)

        # Convenience shape references
        self.shape          = (cfg.n_z, cfg.n_r_ret + 3, cfg.n_c)
        self.gas_shape      = (cfg.n_z, cfg.n_c)
        self.particle_shape = (cfg.n_z, cfg.n_r_ret, cfg.n_c)

        # Grid coordinates (forwarded from TransportOperators for plotting)
        self.z_c    = self.ops.z_c
        self.r_c    = self.ops.r_c_ret

        self.numjac = NumJac(self.shape, axes_diagonals=[0], axes_blocks=[1, 2])
        self.u0 = self._initial_state()
        self.u  = self.u0.copy()

    def _initial_state(self) -> np.ndarray:
        u = np.zeros(self.shape)
        u[:, :-1, :] = self.cfg.inlet_concentration.reshape(1, 1, self.cfg.n_c)
        u[:, -1, :]  = 0.0
        return u

    def split_state(self, u: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        u = u.reshape(self.shape)
        return u[:, 0, :], u[:, 1, :], u[:, 2:-1, :], u[:, -1, :]

    def _particle_average_source(self, c_p: np.ndarray) -> np.ndarray:
        source = particle_reaction_rates(c_p, self.cfg)
        return np.sum(source * self.ops.volume_weights.reshape(1, -1, 1), axis=1)

    def _particle_apparent_source(self, c_p: np.ndarray, c_b: np.ndarray) -> np.ndarray:
        ops = self.ops
        c_p_vec = c_p.reshape(-1, 1)
        c_b_vec = c_b[:, None, :].reshape(-1, 1)
        return (
            ops.particle_apparent_mat    @ c_p_vec
            + ops.particle_apparent_bc_mat @ c_b_vec
        ).reshape(self.gas_shape)

    def residual_values(self, u: np.ndarray) -> np.ndarray:
        cfg = self.cfg
        ops = self.ops
        c_g, c_b, c_p, c_m = self.split_state(u)
        source_particle = particle_reaction_rates(c_p, cfg)
        source_reactor = cfg.eps_s * self._particle_apparent_source(c_p, c_b)
        p_mask = cfg.P_vector.reshape(1, -1)
        membrane_flux = p_mask * (c_g - c_m)
        residual = np.zeros_like(u).reshape(self.shape)

        # [0] Retentate bulk: convection–dispersion – apparent source + membrane loss
        # [1] Boundary layer = bulk gas (no film resistance)
        # [2:-1] Intraparticle diffusion–reaction
        # [-1] Permeate convection + membrane gain
        residual[:, 0, :] = (
            ops.gas_transport_const
            + ops.gas_transport_mat @ c_g.reshape(-1, 1)
        ).reshape(self.gas_shape) - source_reactor + cfg.a_ret * membrane_flux
        
        residual[:, 1, :] = c_b - c_g 
        
        residual[:, 2:-1, :] = (
            ops.particle_diffusion_mat @ c_p.reshape(-1, 1)
            + ops.particle_boundary_mat @ c_b[:, None, :].reshape(-1, 1)
        ).reshape(self.particle_shape) - source_particle

        residual[:, -1, :] = (
            ops.perm_transport_const
            + ops.perm_transport_mat @ c_m.reshape(-1, 1)
        ).reshape(self.gas_shape) - cfg.a_perm * membrane_flux


        return residual

    def residual(self, u: np.ndarray) -> tuple[np.ndarray, np.ndarray]:   
        u = u.reshape(self.shape)
        f = self.residual_values(u)
        f, jac = self.numjac(self.residual_values, u, f_value=f)
        return f.ravel(), jac

    def solve(self):
        result = newton(self.residual, self.u, tol=self.cfg.tol, maxfev=self.cfg.maxfev, solver="spsolve")
        self.u = result.x.reshape(self.shape)
        self.result = result
        return result

    def fields(self,) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        return self.split_state(self.u)

    def effectiveness_profile(self,) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        _c_g, c_b, c_p, _c_m = self.fields()
        r1_int, r2_int = reaction_rates(c_p, self.cfg)
        w = self.ops.volume_weights.reshape(1, -1)

        r1_apparent = np.sum(r1_int * w, axis=1)
        r2_apparent = np.sum(r2_int * w, axis=1)

        r1_surf_full, r2_surf_full = reaction_rates(c_b[:, None, :], self.cfg)
        r1_surface = r1_surf_full[:, 0]   
        r2_surface = r2_surf_full[:, 0]   

        eta_r1 = -r1_apparent / np.maximum(r1_surface, 1.0e-30)
        eta_r2 = -r2_apparent / np.maximum(r2_surface, 1.0e-30)
        return eta_r1, eta_r2, r1_surface, r2_surface


In [37]:
cfg   = ModelConfig()
model = MembraneReactorModel(cfg)
result = model.solve()
c_g, c_b, c_p, c_m = model.fields()

c_g, c_b, c_p, c_m = model.fields()
RT = cfg.R * cfg.T
P_at_outlet = c_p[-1, -1, :] * RT
r1, r2 = reaction_rates(c_b[:, None, :], cfg)


display(pd.Series(
    {
        "success":                           result.success,
        "message":                           result.message,
        "Newton iterations":                 result.nit,
        "final residual norm (inf)":         np.linalg.norm(result.fun, ord=np.inf),
        "outlet CO2 conversion [-]":         1.0 - c_g[-1, 0] / c_g[0, 0],
        "outlet H2 conversion [-]":          1.0 - c_g[-1, 1] / c_g[0, 1],
        "outlet DMC concentration [mol/m³]": c_g[-1, 4],
        "outlet CO2 concentration [mol/m³]": c_g[-1, 0],
        "outlet H2 concentration [mol/m³]":  c_g[-1, 1],
        "outlet CH3OH concentration [mol/m³]": c_g[-1, 2],
        "outlet H2O permeate concentration [mol/m³]": c_m[-1, 3],
    },
    name="settings",
))

display(pd.Series(
    {
        "k_eff_MeOH":                cfg.k_eff_r1(),
        "k_eff_DMC (R2)":            cfg.k_eff_r2(cfg.p),
        "p_CO2 [Pa]":                P_at_outlet[0],
        "p_H2 [Pa]":                 P_at_outlet[1],
        "p_CH3OH [Pa]":              P_at_outlet[2],
        "p_H2O [Pa]":                P_at_outlet[3],
        "p_DMC [Pa]":                P_at_outlet[4],
        "r1 [mol/m³/s]":             r1[-1, 0],
        "r2 [mol/m³/s]":             r2[-1, 0],
    },

    name="outlet diagnostics",

))


AttributeError: 'ModelConfig' object has no attribute 'rho_bulk'

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 8), sharex=False)
axes = axes.flatten()

# axes[0]: Bulk Gas Concentration
for i, label in enumerate(model.species_labels):
    axes[0].plot(model.z_c, c_g[:, i], label=label, linewidth=2)
axes[0].axhline(0.0, color="k", linewidth=0.8)
axes[0].set_xlabel("z [m]")
axes[0].set_ylabel("Concentration [mol m$^{-3}$]")
axes[0].set_title("Bulk gas concentration")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), ncol=1, fontsize=8, frameon=True)
axes[0].grid(True, linestyle=":", alpha=0.6)

# axes[1]: H2O Permeate Concentration
for i, label in enumerate(model.species_labels):
    axes[1].plot(model.z_c, c_m[:, i], label=label, linewidth=2)
axes[1].axhline(0.0, color="k", linewidth=0.8)
axes[1].set_xlabel("z [m]")
axes[1].set_ylabel(r"Concentration [mol m$^{-3}$]")
axes[1].set_title("H2O Permeate concentration")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), ncol=1, fontsize=8, frameon=True)
axes[1].grid(True, linestyle=":", alpha=0.6)

# axes[2]: Reaction Rates
axes[2].plot(model.z_c, r1, label=r"$r1$", linewidth=2)
axes[2].plot(model.z_c, r2, label=r"$r2$", linewidth=2)
axes[2].axhline(0.0, color="k", lw=0.8)
axes[2].set_xlabel("z [m]")
axes[2].set_ylabel("Rate [mol m$^{-3}$ s$^{-1}$]")
axes[2].set_title("Reaction rates at pellet boundary")
axes[2].legend(loc="best", fontsize=8)
axes[2].grid(True, linestyle=":", alpha=0.6)

# axes[3]: Effectiveness Profile 
eta_r1, eta_r2, _, _ = model.effectiveness_profile()
axes[3].plot(model.z_c, eta_r1, label=r"$\eta$ R1", linewidth=2)
axes[3].plot(model.z_c, eta_r2, label=r"$\eta$ R2", linewidth=2)
axes[3].axhline(0.0, color="k", linewidth=0.8, linestyle="--")
axes[3].set_xlabel("z [m]")
axes[3].set_ylabel("Effectiveness factor")
axes[3].set_title(r"Effectiveness ($\eta$)")
axes[3].grid(True, linestyle=":", alpha=0.6)
axes[3].legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()